# Data

*\[\[\[model, 123\]\]\]*

In [1]:
!ls yugioh

ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl


In [ ]:
!conda install -c anaconda python=3.8

Solving environment: failed with current_repodata.json, will retry with next repodata source.
Initial quick solve with frozen env failed.  Unfreezing env and trying again.
Solving environment: failed with current_repodata.json, will retry with next repodata source.
Solving environment: failed
Initial quick solve with frozen env failed.  Unfreezing env and trying again.
Solving environment: - 

In [15]:
!pip show scikit-learn

Name: scikit-learn
Version: 1.0.2
Summary: A set of python modules for machine learning and data mining
Home-page: http://scikit-learn.org
Author: 
Author-email: 
License: new BSD
Location: /home/shevkunov/anaconda3/lib/python3.7/site-packages
Requires: joblib, numpy, scipy, threadpoolctl
Required-by: seqeval


In [1]:
import pickle
import numpy as np
import os
import tqdm
import torch
import gc

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

from importlib import reload
from matplotlib import pyplot as plt

import matrix_approx_zeshel

matrix_approx_zeshel = reload(matrix_approx_zeshel)

!ls  | grep matrix_approx_zeshel

2023-09-09 13:56:05.519276: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-09-09 13:56:05.550155: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-09-09 13:56:05.673585: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-09-09 13:56:05.674458: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-09-09 13:56:06.319380: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Co

matrix_approx_zeshel.py


# Open Data loader & context

In [2]:
def load_ment_to_ent_scores(directory = "yugioh", shuffle_rows = 0, full = True):
    data = list()

    for file in os.listdir(directory):
        path = f"{directory}/{file}"
        print(f"Loading file {path}")
        with open(path, "rb") as f:
            data.append(
                pickle.load(f)
            )
    data = sorted(data, key = lambda x: x["arg_dict"]["n_ment_start"])

    for i in range(len(data) - 1):
        if full:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] == data[i + 1]["arg_dict"]["n_ment_start"]
        else:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] <= data[i + 1]["arg_dict"]["n_ment_start"]
        
    ment_to_ent_scores = list(map(lambda x: x["ment_to_ent_scores"], data))
    ment_to_ent_scores = np.vstack(ment_to_ent_scores)
    print("Loaded shape = ", ment_to_ent_scores.shape)
    
    if shuffle_rows:
        print(f"Shuffling... (seed = {shuffle_rows})")
        np.random.seed(shuffle_rows)
        np.random.shuffle(ment_to_ent_scores)
    
    return ment_to_ent_scores

In [3]:
from sklearn.cluster import KMeans

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

class EvalContextRelevs:
    def __init__(self, relevs, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
        self.relevs = np.array(relevs)
        self.reqs_count = self.relevs.shape[0]
        self.games_count = self.relevs.shape[1]
        
        self.key_size = key_size
        self.key_games = (
            np.random.choice(np.arange(self.games_count), key_size, replace=False)
            if key_games is None else
            key_games
        )
        np.random.seed(seed)
        #np.random.shuffle(self.requests)

        self.try_det_attempts(det_attempts)
        self.train_split = int(self.reqs_count * train_size)

        assert key_size + 1 < self.train_split


        self.key_relevs = self.relevs[:key_size]
        self.train_relevs = self.relevs[key_size + 1: self.train_split]
        self.test_relevs = self.relevs[self.train_split:]

        self.slices = ["key", "train", "test"]
        print(len(self.key_relevs), len(self.train_relevs), len(self.test_relevs))

    def get_top_games(self):
        return self.relevs.mean(axis=0).argsort()[:100]

    def set_top_games_as_key(self):
        self.key_games = self.get_top_games()
        return self

    def get_kmeans_games(self, all_from_labels=True):
        X = self.get_relevs("train").T

        k_func = (
            (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
            if not all_from_labels else
            (lambda C : from_labels(X, C.labels_))
        )
        K_KMeans = k_func(
            KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
        )
        
        return K_KMeans
    
    def set_kmeans_games_as_key(self, *args, **kwargs):
        self.key_games = self.get_kmeans_games(*args, **kwargs)
        return self
    

    def try_det_attempts(self, det_attempts):
        def get_det(r):
            kr = r[:self.key_size, self.key_games] - r.mean()
            return np.abs(np.linalg.det(kr))

        best_i_array = np.arange(len(self.relevs))

        for i in range(det_attempts):

            r_i_array = np.arange(len(self.relevs))
            np.random.shuffle(r_i_array)

            n, o = get_det(self.relevs[r_i_array, :]), get_det(self.relevs[best_i_array, :])
            
            # print(f"try update key_reqs ({o} vs {n}...")
            if n > o:
                best_i_array = r_i_array
                print(f"updated det ({i}, {o} -> {n})")

        print("Best det = ", get_det(self.relevs[best_i_array, :]))

        self.relevs = self.relevs[best_i_array, :]
        print("Current de = ", get_det(self.relevs))
        
    def get_relevs(self, t = "train"):
        if t == "train":
            return self.train_relevs
        elif t == "key":
            return self.key_relevs
        elif t == "test":
            return self.test_relevs
        else:
            assert False
            
    def get_requests(self, t = "train"):
        if t == "train":
            return self.train_reqs
        elif t == "key":
            return self.key_reqs
        elif t == "test":
            return self.test_reqs
        else:
            assert False

In [4]:
# ctx = EvalContextRelevs(load_ment_to_ent_scores("yugioh", shuffle_rows=42), det_attempts=100)

# Games Data loader & context

In [5]:
import collections
import pickle
import numpy as np
import tqdm
import os
import gc

import matplotlib.pyplot as plt


def load(limit, raw_path = "stand/log.local.logtime2.txt", path = "log.local.logtime2.bin", key_games = None, seed = 17, det_attempts = 0):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            flimit, frequests, fdocembs = pickle.load(f)
            if flimit == limit:
                requests, docembs = frequests, fdocembs
            else:
                print(f"WARN: buffered limit is different, {flimit} != {limit}, reloading...")

    if not requests:
        with open(raw_path) as f:
            req = list()
            reqid = None
            models = list()
            prevreqmodel = None
            reqmodel = dict()
            prevmodelid = -1
            bannermodelid = -1
            for i, line in tqdm.tqdm_notebook(enumerate(f)):
                if line.startswith("Model = 6;"):
                    prevreqmodel = reqmodel
                    reqmodel = dict()

                if line.startswith("Model = "):
                    spl = line.split(" ")
                    prevmodelid = int(spl[2][:-1])
                    bannermodelid = max(bannermodelid , prevmodelid)
                    reqmodel[prevmodelid] = readvector(spl[3])
                elif line.startswith("dbid"):
                    spl = line.split(" ")
                    dbid = int(spl[1][:-1])
                    docembs[bannermodelid][dbid] = readvector(spl[2])
                elif line.startswith("seed"):
                    if len(requests) >= limit:
                        break
                    if req:
                        requests.append((reqid, prevreqmodel, sorted(req)))
                        req = list()
                    reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
                else:
                    req.append(
                        (int(line.split()[0]), float(line.split()[1]))
                    )
        
        with open(path, "wb") as f:
            pickle.dump((limit, requests, docembs), f)

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
            self.games_count = games_count
            
            self.key_size = key_size
            self.key_games = (
                np.random.choice(np.arange(games_count), key_size, replace=False)
                if key_games is None else
                key_games
            )

            self.requests = requests
            np.random.seed(seed)
            np.random.shuffle(self.requests)
            
            self.try_det_attempts(det_attempts)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks
            self.relevs = dict()
            
        def get_top_games(self):
            if not hasattr(self, "top_games"):
                embed_games = np.array([
                    np.array([r[2][g_i][1] for r in self.get_requests("train")])
                    for g_i in range(self.games_count)
                ])

                self.embed_games_mean = embed_games.mean(axis=1)
                self.top_games_all = (-self.embed_games_mean).argsort()
                self.top_games = self.top_games_all[:len(self.key_games)]

            return self.top_games
        
        def set_top_games_as_key(self):
            self.key_games = self.get_top_games()
            return self
        
        def get_kmeans_games(self, all_from_labels=True):
            X = self.get_relevs("train").T

            k_func = (
                (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
                if not all_from_labels else
                (lambda C : from_labels(X, C.labels_))
            )
            K_KMeans = k_func(
                KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
            )

            return K_KMeans

        def set_kmeans_games_as_key(self, *args, **kwargs):
            self.key_games = self.get_kmeans_games(*args, **kwargs)
            return self

        def try_det_attempts(self, det_attempts, model_id = 6):
            def get_det(r, r_i_array):
                kr = np.array([
                    r[r_i][1][model_id]
                    for r_i in r_i_array[:100]
                ])
                return np.abs(np.linalg.det(kr[:kr.shape[1], :]))

            best_i_array = np.arange(len(self.requests))

            for _ in range(det_attempts):
                # print("try update key_reqs...")
                
                r_i_array = np.arange(len(self.requests))
                np.random.shuffle(r_i_array)
                
                n, o = get_det(self.requests, r_i_array), get_det(self.requests, best_i_array)
                # print(n, o)
                if n > o:
                    best_i_array = r_i_array
                    # print("updated!")

            print("Best det = ", get_det(self.requests, best_i_array))
            
            new_requests = [
                self.requests[i]
                for i in best_i_array
            ]
            
            del self.requests
            gc.collect()

            self.requests = new_requests
            print(get_det(self.requests, np.arange(len(self.requests))))

        def get_relevs(self, t = "train"):
            if t not in self.relevs:
                self.relevs[t] = np.array([
                    np.array([g_i[1] for g_i in r[2]])
                    for r in self.get_requests(t)
                ])
                
            return self.relevs[t]

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext(key_games = key_games, seed = seed, det_attempts = det_attempts)

In [6]:
# ctx = load(L, raw_path = "stand/log.local.txt", seed=17, det_attempts=DA).set_top_games_as_key()

# Models

In [7]:
class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

In [8]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "matrix_l2": 0,
    "dssm_l2": 0,
    "train_popbias": False,
    "train_bias": False,
    "train_vbias": False,
    "verbose": True,
    "train_matrix": True,
    "train_dssm": False,
    "loss": "mse",
    "ubatch": 1e9,
    "activation": "relu",
    "score_verbose": 0,
    "trainable_items": False,
    "use_keys_in_train": False
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(fit_kwargs)
        self.fit_kwargs = p
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[i] for i in ctx.key_games])
                    for r_i in ctx.get_relevs(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[g_i] for r in ctx.get_relevs("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        
        if self.fit_kwargs["trainable_items"]:
            self.trainable_games = tf.Variable(
                tf.convert_to_tensor(self.embed_games, dtype=tf.float32),
                trainable=True
            )
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = ctx.get_relevs("key")
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        return (self.embed_users[t] - self.embed_users_mean["train"]) / self.embed_users[t].std(axis=0)
        # return self.embed_users[t]
    
    def get_game_embs(self):
        if not self.fit_kwargs["trainable_items"]:
            return (self.embed_games - self.embed_games.mean(axis=0)) / self.embed_games.std(axis=0)
        else:
            return self.trainable_games
            
        #return self.embed_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.p = p
        self.train_bias = p["train_bias"]
        self.train_vbias = p["train_vbias"]
        self.train_popbias = p["train_popbias"]
        self.train_matrix = p["train_matrix"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        if p["use_keys_in_train"]:
            train_user_scores = np.vstack([
                self.get_user_scores("key"),
                self.get_user_scores("train")
            ])
            train_user_embs = np.vstack([
                self.get_user_embs("key"),
                self.get_user_embs("train")
            ])
        else:
            train_user_scores = self.get_user_scores("train")
            train_user_embs = self.get_user_embs("train")

        game_embs = self.get_game_embs()
        
        initializer = tf.keras.initializers.GlorotUniform()
        values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        self.W = tf.Variable(values / 100., trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        if self.train_vbias:
            self.vb = tf.Variable(
                np.zeros_like(self.game_avg_scores["train"], dtype=np.float32),
                trainable = True
            ) 
        
        
        for i in range(p["n"]):
            def loss():
                def get_logits_scores(loss_batch = 1e9):
                    game_slice = None
                    
                    ubatch = p["ubatch"]
                    if ubatch >= train_user_scores.shape[0]:
                        train_user_embs_ = train_user_embs
                        scores_ = train_user_scores
                    else:
                        u_slice = np.random.choice(np.arange(train_user_scores.shape[0]), ubatch, replace = True)
                        train_user_embs_ = train_user_embs[u_slice]
                        scores_ = train_user_scores[u_slice]
                    
                    if loss_batch >= game_embs.shape[0]:
                        game_embs_ = game_embs
                        scores_ = scores_
                    else:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        game_embs_ = (
                            game_embs[game_slice]
                            if isinstance(game_embs, np.ndarray) else
                            tf.gather(game_embs, game_slice)
                        )
                        scores_ = scores_[:, game_slice]
                        
                    
                    logits = 0
                    
                    if self.train_matrix:
                        logits += train_user_embs_ @ self.W @ game_embs_.T

                    if self.train_dssm:
                        logits += self.u_dssm(train_user_embs_) @ tf.transpose(self.g_dssm(game_embs_))
                    
                    if self.train_popbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.pb * self.game_avg_scores["train"]
                        else:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_vbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.vb
                        else:
                            logits += tf.gather(self.vb, game_slice)
                        
                    if self.train_bias:
                        logits += self.b
                        
                    return logits, scores_
                        
                if self.loss == "mse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    v = tf.reduce_mean((scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    q_mean = scores.mean(axis=1)
                    v = tf.reduce_mean(((scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    while True:
                        logits, scores = get_logits_scores(
                            loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            scores.astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                elif self.loss == "softmax_signed":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    sf = tf.nn.softmax(logits, axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        sf,
                        -sf
                    ))
                elif self.loss == "softmax_weighted":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ) * scores)
                elif self.loss == "sigmoid_top_100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    mask = np.argsort(-scores, axis=1) < 100
                    v = -tf.reduce_sum(
                        tf.math.log_sigmoid((2 * mask - 1) * logits)
                    )
                elif self.loss == "KL100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    mask = np.argsort(-scores, axis=1) < 100
                    v = tf.keras.losses.KLDivergence()((scores.T >= qs.T).T, logits)
                else:
                    assert False
                    
                if self.p["c"]:
                    v += tf.reduce_mean(self.W * self.W) * p["c"]
                    
                if self.p["dssm_l2"]:
                    for weights_ in [self.u_dssm.weights, self.g_dssm.weights]:
                        for weight_ in weights_:
                            v += tf.reduce_sum(weight_ * weight_) * self.p["dssm_l2"]
                
                if self.p["matrix_l2"]:
                    v += tf.reduce_sum(self.W * self.W) * self.p["matrix_l2"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            
            if self.train_matrix:
                weights += [self.W]

            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_vbias:
                weights += [self.vb]   
                
            if self.train_bias:
                weights += [self.b]   
                
            
            if self.p["trainable_items"]:
                weights += [self.trainable_games]
                
                
            opt.minimize(loss, weights)
            
            if p["score_verbose"] and (i % p["score_verbose"] == 0):
                print(f"\n=== Iteration {i} ===")
                for sl in self.ctx.slices:
                    print(f"slice = {sl}, score = {self.get_score(sl)}")
        print("last loss = ", loss())

    def recommend(self, t):
        logits = 0
                    
        if self.train_matrix:
            logits += self.get_user_embs(t) @ self.W @ tf.transpose(self.get_game_embs())

        if self.train_dssm:
            logits += self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_vbias:
            logits += self.vb
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [9]:
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [10]:
class AnnCUR(Popular):
    def __init__(self, ctx, oracle=False, key_games=None, name=None):
        super().__init__(ctx)
        self.oracle = oracle
        self.ctx = ctx
        self.name = name

        self.key_cols_idx = np.array(sorted(ctx.key_games if key_games is None else key_games))
        rows_idx = np.arange(ctx.get_relevs("train").shape[0])
        
        self.cur = matrix_approx_zeshel.CURApprox(
            rows=torch.from_numpy(ctx.get_relevs("train")),
            cols=torch.from_numpy(ctx.get_relevs("train")[:, self.key_cols_idx]),
            row_idxs=rows_idx,
            col_idxs=self.key_cols_idx,
            approx_preference="rows",
            A=(torch.from_numpy(ctx.get_relevs("train")) if oracle else None)
        )        
        
    def __repr__(self):
        if self.name is None:
            return f"AnnCur({id(self)})"
        else:
            return f"AnnCur({self.name} - {id(self)})"
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        key_relevs = torch.from_numpy(self.ctx.get_relevs(t)[:, self.key_cols_idx])
        return self.cur.get_complete_row(key_relevs)
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

# Evals

### open

In [11]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

def K_by_min(X):
    center = X.mean(axis=0)
    K = [euclidean_distances(X, center.reshape(1, -1)).argmax()]

    while len(K) < 100:
        K.append(euclidean_distances(X, X[K]).min(axis=1).argmax())
    return K


def eval_clustering(ctx, all_from_labels=False):
    X = ctx.get_relevs("train").T
    from sklearn.cluster import KMeans
    
    k_func = (
        (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
        if not all_from_labels else
        (lambda C : from_labels(X, C.labels_))
    )
    
    K_KMeans = k_func(
        KMeans(n_clusters=100, random_state=0).fit(X)  #, n_init="auto").fit(X)
    )


    from sklearn.cluster import BisectingKMeans
    K_BisectingKMeans = k_func(
        BisectingKMeans(n_clusters=100, random_state=0).fit(X)
    )


    from sklearn.cluster import MiniBatchKMeans
    K_MiniBatchKMeans = from_labels(
        X,
        MiniBatchKMeans(n_clusters=100, random_state=0).fit(X).labels_
    )

    from sklearn.cluster import AgglomerativeClustering
    K_AgglomerativeClustering = from_labels(
        X,
        AgglomerativeClustering(n_clusters=100).fit(X).labels_
    )

    from sklearn.cluster import SpectralCoclustering
    K_SpectralCoclustering = from_labels(
        X,
        SpectralCoclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralBiclustering
    K_SpectralBiclustering = from_labels(
        X,
        SpectralBiclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralClustering
    K_SpectralClusteringNN = from_labels(
        X,
        SpectralClustering(n_clusters=100, random_state=0, affinity="nearest_neighbors", n_jobs=-1).fit(X).labels_
    )

    K_ByMin = K_by_min(X)

    ev([
        Popular(ctx),
        AnnCUR(ctx),
        AnnCUR(ctx, key_games=np.arange(100), name="key_games=np.arange(100)"),
        AnnCUR(ctx, key_games=np.random.choice(np.arange(X.shape[0]), size=100, replace=False), name="random"),
        AnnCUR(ctx, key_games=K_KMeans, name="K_KMeans"),
        AnnCUR(ctx, key_games=K_BisectingKMeans, name="K_BisectingKMeans"),
        AnnCUR(ctx, key_games=K_MiniBatchKMeans, name="K_MiniBatchKMeans"),
        AnnCUR(ctx, key_games=K_AgglomerativeClustering, name="K_AgglomerativeClustering"),
        AnnCUR(ctx, key_games=K_SpectralCoclustering, name="K_SpectralCoclustering"),
        AnnCUR(ctx, key_games=K_SpectralBiclustering, name="K_SpectralBiclustering"),
        AnnCUR(ctx, key_games=K_SpectralClusteringNN, name="K_SpectralClusteringNN"),
        AnnCUR(ctx, key_games=K_ByMin, name="K_ByMin"),
    ])

In [12]:
DATASETS = ["yugioh", "pro_wrestling", "star_trek", "doctor_who", "military"]

In [23]:
dataset = DATASETS[0]
ctx = EvalContextRelevs(
    load_ment_to_ent_scores(dataset, shuffle_rows=42, full=(dataset != "military")),
    det_attempts=100
)

Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013


In [13]:
ev([
    AnnCUR(ctx),
    AnnCUR(ctx, key_games=np.arange(100), name="key_games=np.arange(100)"),
])

Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013



model =  AnnCur(140671859251264)
np.mean(results), mse, len(results) =  0.4881150442477877 0.18208094 2260
np.mea

In [24]:
ctx.set_kmeans_games_as_key()

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [15]:
ev([
    AnnCUR(ctx),
])




model =  AnnCur(140671859253472)
np.mean(results), mse, len(results) =  0.5300929203539823 0.14258455 2260
np.mean(results), mse, len(results) =  0.5100394866732478 0.16796441 1013
0.5300929203539823 0.5100394866732478


In [18]:
N = 20000

m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': False,
    'verbose': False, 'loss': 'softmax',
    'loss_batch': 128, 'loss_q': 0.99,
    'n': N,
    # 'ubatch': 512
    'score_verbose': 1000
})
m.fit()
m.get_score("train"), m.get_score("test")
del m

self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 10031)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (2260, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0819 24.044302 100
slice = key, score = 0.0819
np.mean(results), mse, len(results) =  0.07364159292035398 25.591763 2260
slice = train, score = 0.07364159292035398
np.mean(results), mse, len(results) =  0.0722704837117473 25.203623 1013
slice = test, score = 0.0722704837117473

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.5514 399.37567 100
slice = key, score = 0.5514
np.mean(results), mse, len(results) =  0.46561061946902654 416.1297 2260
slice = train, score = 0.46561061946902654
np.mean(results), mse, len(results) =  0.4451530108588352 412.7993 1013
slice = test, score = 0.4451530108588352

=== Iteration 2000 ===
np.mean(results), mse, len(results) =  0.5754 775.2

In [25]:
N = 20000

m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': False,
    'verbose': False, 'loss': 'softmax',
    'loss_batch': 128, 'loss_q': 0.99,
    'n': N,
    # 'ubatch': 512
    'score_verbose': 1000,
    'trainable_items': True  # <<< DIFF HERE
})
m.fit()
m.get_score("train"), m.get_score("test")
del m

self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 10031)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (2260, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.09140000000000002 714.29846 100
slice = key, score = 0.09140000000000002
np.mean(results), mse, len(results) =  0.08863716814159292 744.66766 2260
slice = train, score = 0.08863716814159292
np.mean(results), mse, len(results) =  0.08751233958538994 715.69604 1013
slice = test, score = 0.08751233958538994

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.5479999999999999 11722.787 100
slice = key, score = 0.5479999999999999
np.mean(results), mse, len(results) =  0.453070796460177 11234.573 2260
slice = train, score = 0.453070796460177
np.mean(results), mse, len(results) =  0.436791707798618 11919.683 1013
slice = test, score = 0.436791707798618

=== Iteration 2000 ===
np.

In [19]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000
})
m.fit()
m.get_score("train"), m.get_score("test")
del m

self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 10031)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (2260, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0837 42.56024 100
slice = key, score = 0.0837
np.mean(results), mse, len(results) =  0.06432743362831858 38.621704 2260
slice = train, score = 0.06432743362831858
np.mean(results), mse, len(results) =  0.06554787759131293 39.652866 1013
slice = test, score = 0.06554787759131293

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.5611999999999999 444.46072 100
slice = key, score = 0.5611999999999999
np.mean(results), mse, len(results) =  0.4584823008849558 446.2885 2260
slice = train, score = 0.4584823008849558
np.mean(results), mse, len(results) =  0.4435439289239881 433.85806 1013
slice = test, score = 0.4435439289239881

=== Iteration 2000 ===
np.mean(results), mse, len(

np.mean(results), mse, len(results) =  0.48597235932872657 7809.6753 1013
slice = test, score = 0.48597235932872657

=== Iteration 23000 ===
np.mean(results), mse, len(results) =  0.5935 7948.495 100
slice = key, score = 0.5935
np.mean(results), mse, len(results) =  0.5329823008849557 8342.372 2260
slice = train, score = 0.5329823008849557
np.mean(results), mse, len(results) =  0.4869397828232972 8195.025 1013
slice = test, score = 0.4869397828232972

=== Iteration 24000 ===
np.mean(results), mse, len(results) =  0.6005000000000001 7827.276 100
slice = key, score = 0.6005000000000001
np.mean(results), mse, len(results) =  0.5338495575221239 8086.215 2260
slice = train, score = 0.5338495575221239
np.mean(results), mse, len(results) =  0.4883415597235933 7954.221 1013
slice = test, score = 0.4883415597235933

=== Iteration 25000 ===
np.mean(results), mse, len(results) =  0.5982000000000001 8350.403 100
slice = key, score = 0.5982000000000001
np.mean(results), mse, len(results) =  0.53468

np.mean(results), mse, len(results) =  0.545137168141593 22787.68 2260
slice = train, score = 0.545137168141593
np.mean(results), mse, len(results) =  0.4916979269496545 22523.182 1013
slice = test, score = 0.4916979269496545

=== Iteration 47000 ===
np.mean(results), mse, len(results) =  0.5923000000000002 21918.553 100
slice = key, score = 0.5923000000000002
np.mean(results), mse, len(results) =  0.5433716814159292 23579.797 2260
slice = train, score = 0.5433716814159292
np.mean(results), mse, len(results) =  0.4899605133267523 23052.002 1013
slice = test, score = 0.4899605133267523

=== Iteration 48000 ===
np.mean(results), mse, len(results) =  0.5831000000000001 22704.03 100
slice = key, score = 0.5831000000000001
np.mean(results), mse, len(results) =  0.5435044247787612 23888.188 2260
slice = train, score = 0.5435044247787612
np.mean(results), mse, len(results) =  0.4904738400789733 23585.959 1013
slice = test, score = 0.4904738400789733

=== Iteration 49000 ===
np.mean(results), 


=== Iteration 70000 ===
np.mean(results), mse, len(results) =  0.5823999999999999 42497.516 100
slice = key, score = 0.5823999999999999
np.mean(results), mse, len(results) =  0.5468495575221239 45390.625 2260
slice = train, score = 0.5468495575221239
np.mean(results), mse, len(results) =  0.48848963474827245 45025.973 1013
slice = test, score = 0.48848963474827245

=== Iteration 71000 ===
np.mean(results), mse, len(results) =  0.5838 42627.418 100
slice = key, score = 0.5838
np.mean(results), mse, len(results) =  0.5489690265486726 44423.223 2260
slice = train, score = 0.5489690265486726
np.mean(results), mse, len(results) =  0.4909871668311945 43956.0 1013
slice = test, score = 0.4909871668311945

=== Iteration 72000 ===
np.mean(results), mse, len(results) =  0.5840000000000001 43039.098 100
slice = key, score = 0.5840000000000001
np.mean(results), mse, len(results) =  0.5530575221238938 45583.406 2260
slice = train, score = 0.5530575221238938
np.mean(results), mse, len(results) =  0

np.mean(results), mse, len(results) =  0.4929516288252715 72836.28 1013
slice = test, score = 0.4929516288252715

=== Iteration 94000 ===
np.mean(results), mse, len(results) =  0.5844999999999999 71559.86 100
slice = key, score = 0.5844999999999999
np.mean(results), mse, len(results) =  0.5560486725663717 77894.34 2260
slice = train, score = 0.5560486725663717
np.mean(results), mse, len(results) =  0.4936130306021718 77114.38 1013
slice = test, score = 0.4936130306021718

=== Iteration 95000 ===
np.mean(results), mse, len(results) =  0.5760000000000001 71546.77 100
slice = key, score = 0.5760000000000001
np.mean(results), mse, len(results) =  0.5450752212389381 76470.33 2260
slice = train, score = 0.5450752212389381
np.mean(results), mse, len(results) =  0.48417571569595264 75994.16 1013
slice = test, score = 0.48417571569595264

=== Iteration 96000 ===
np.mean(results), mse, len(results) =  0.5848 73446.74 100
slice = key, score = 0.5848
np.mean(results), mse, len(results) =  0.551088

In [26]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': True  # <<< DIFF HERE
})
m.fit()
m.get_score("train"), m.get_score("test")
del m

self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 10031)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (2260, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0513 8861.822 100
slice = key, score = 0.0513
np.mean(results), mse, len(results) =  0.05354424778761062 8378.432 2260
slice = train, score = 0.05354424778761062
np.mean(results), mse, len(results) =  0.05339585389930899 8435.37 1013
slice = test, score = 0.05339585389930899

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.541 16624.756 100
slice = key, score = 0.541
np.mean(results), mse, len(results) =  0.4476504424778761 17817.818 2260
slice = train, score = 0.4476504424778761
np.mean(results), mse, len(results) =  0.43395853899308984 18647.254 1013
slice = test, score = 0.43395853899308984

=== Iteration 2000 ===
np.mean(results), mse, len(results) =  0.563000000000


=== Iteration 23000 ===
np.mean(results), mse, len(results) =  0.5817 117102.805 100
slice = key, score = 0.5817
np.mean(results), mse, len(results) =  0.592216814159292 110825.53 2260
slice = train, score = 0.592216814159292
np.mean(results), mse, len(results) =  0.5149654491609083 114407.71 1013
slice = test, score = 0.5149654491609083

=== Iteration 24000 ===
np.mean(results), mse, len(results) =  0.5859999999999999 103658.164 100
slice = key, score = 0.5859999999999999
np.mean(results), mse, len(results) =  0.5929867256637169 115163.58 2260
slice = train, score = 0.5929867256637169
np.mean(results), mse, len(results) =  0.5158242843040474 116582.445 1013
slice = test, score = 0.5158242843040474

=== Iteration 25000 ===
np.mean(results), mse, len(results) =  0.5805 107975.25 100
slice = key, score = 0.5805
np.mean(results), mse, len(results) =  0.5952168141592921 104356.79 2260
slice = train, score = 0.5952168141592921
np.mean(results), mse, len(results) =  0.5156268509378086 11132

np.mean(results), mse, len(results) =  0.5219940769990128 292722.0 1013
slice = test, score = 0.5219940769990128

=== Iteration 47000 ===
np.mean(results), mse, len(results) =  0.5753999999999999 305480.8 100
slice = key, score = 0.5753999999999999
np.mean(results), mse, len(results) =  0.6207300884955752 303587.38 2260
slice = train, score = 0.6207300884955752
np.mean(results), mse, len(results) =  0.5202566633761105 314097.1 1013
slice = test, score = 0.5202566633761105

=== Iteration 48000 ===
np.mean(results), mse, len(results) =  0.5774 300804.16 100
slice = key, score = 0.5774
np.mean(results), mse, len(results) =  0.6232256637168142 292409.34 2260
slice = train, score = 0.6232256637168142
np.mean(results), mse, len(results) =  0.5220138203356367 311435.7 1013
slice = test, score = 0.5220138203356367

=== Iteration 49000 ===
np.mean(results), mse, len(results) =  0.5705 307679.78 100
slice = key, score = 0.5705
np.mean(results), mse, len(results) =  0.6191238938053099 311719.8 22

np.mean(results), mse, len(results) =  0.6382522123893806 482212.47 2260
slice = train, score = 0.6382522123893806
np.mean(results), mse, len(results) =  0.5252714708785784 502057.03 1013
slice = test, score = 0.5252714708785784

=== Iteration 71000 ===
np.mean(results), mse, len(results) =  0.5759 496332.9 100
slice = key, score = 0.5759
np.mean(results), mse, len(results) =  0.6390353982300885 485479.2 2260
slice = train, score = 0.6390353982300885
np.mean(results), mse, len(results) =  0.5236327739387957 509935.62 1013
slice = test, score = 0.5236327739387957

=== Iteration 72000 ===
np.mean(results), mse, len(results) =  0.5715000000000001 539288.25 100
slice = key, score = 0.5715000000000001
np.mean(results), mse, len(results) =  0.6391769911504426 551245.9 2260
slice = train, score = 0.6391769911504426
np.mean(results), mse, len(results) =  0.5244126357354393 576131.25 1013
slice = test, score = 0.5244126357354393

=== Iteration 73000 ===
np.mean(results), mse, len(results) =  0.


=== Iteration 94000 ===
np.mean(results), mse, len(results) =  0.5726 690774.06 100
slice = key, score = 0.5726
np.mean(results), mse, len(results) =  0.6514646017699115 685451.5 2260
slice = train, score = 0.6514646017699115
np.mean(results), mse, len(results) =  0.5278084896347484 708723.2 1013
slice = test, score = 0.5278084896347484

=== Iteration 95000 ===
np.mean(results), mse, len(results) =  0.5685 646580.2 100
slice = key, score = 0.5685
np.mean(results), mse, len(results) =  0.6510840707964601 694677.75 2260
slice = train, score = 0.6510840707964601
np.mean(results), mse, len(results) =  0.5253998025666338 728871.25 1013
slice = test, score = 0.5253998025666338

=== Iteration 96000 ===
np.mean(results), mse, len(results) =  0.5715999999999999 798917.75 100
slice = key, score = 0.5715999999999999
np.mean(results), mse, len(results) =  0.6502654867256636 767107.6 2260
slice = train, score = 0.6502654867256636
np.mean(results), mse, len(results) =  0.5250542941757157 791337.44 

In [28]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )
    
    ctx.set_kmeans_games_as_key()
    
    N = 20000

    m = CBKnnV0(ctx, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': True,
        'train_popbias': True, 'train_bias': False,
        'verbose': False, 'loss': 'softmax',
        'loss_batch': 128, 'loss_q': 0.99,
        'n': N,
        # 'ubatch': 512
        'score_verbose': 1000,
        'trainable_items': True  # <<< DIFF HERE
    })
    m.fit()
    m.get_score("train"), m.get_score("test")
    
    
    del ctx
    gc.collect()

    del m
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013


/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 10031)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (2260, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0783 1574.0903 100
slice = key, score = 0.0783
np.mean(results), mse, len(results) =  0.08281858407079647 1446.7064 2260
slice = train, score = 0.08281858407079647
np.mean(results), mse, len(results) =  0.08375123395853899 1565.4125 1013
slice = test, score = 0.08375123395853899

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.5381999999999999 15985.602 100
slice = key, score = 0.5381999999999999
np.mean(results), mse, len(results) =  0.44672566371681416 16362.959 2260
slice = train, score = 0.44672566371681416
np.mean(results), mse, len(results) =  0.4304244817374136 16431.006 1013
slice = test, score = 0.4304244817374136

=== Iteration 2000 ===
np.mean(results), mse, 

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


self.embed_users['train'].shape =  (873, 100)
self.embed_games.shape =  (10133, 100)
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 10133)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (873, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.14919999999999997 9878.811 100
slice = key, score = 0.14919999999999997
np.mean(results), mse, len(results) =  0.15256586483390608 8189.677 873
slice = train, score = 0.15256586483390608
np.mean(results), mse, len(results) =  0.1635645933014354 8992.268 418
slice = test, score = 0.1635645933014354

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.45990000000000003 107305.54 100
slice = key, score = 0.45990000000000003
np.mean(results), mse, len(results) =  0.4174684994272624 96337.16 873
slice = train, score = 0.4174684994272624
np.mean(results), mse, len(results) =  0.3963397129186603 124702.33 418
slice = test, score = 0.3963397129186603

=== Iteration 2000 ===
np.mean(r

...

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


self.embed_users['train'].shape =  (2699, 100)
self.embed_games.shape =  (40281, 100)
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 40281)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (2699, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0482 3551.1726 100
slice = key, score = 0.0482
np.mean(results), mse, len(results) =  0.04190811411633939 3696.5364 2699
slice = train, score = 0.04190811411633939
np.mean(results), mse, len(results) =  0.04041666666666666 4121.946 1200
slice = test, score = 0.04041666666666666

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.20190000000000002 988235.6 100
slice = key, score = 0.20190000000000002
np.mean(results), mse, len(results) =  0.1344090403853279 749943.25 2699
slice = train, score = 0.1344090403853279
np.mean(results), mse, len(results) =  0.13339166666666666 771166.5 1200
slice = test, score = 0.13339166666666666

=== Iteration 2000 ===
np.mean(results), mse, l

Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1300.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1600.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0000.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2000.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1100.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1000.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0400.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0800.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0600.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2100.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_l

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


self.embed_users['train'].shape =  (1579, 100)
self.embed_games.shape =  (104520, 100)
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 104520)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1579, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.07660000000000002 10767.727 100
slice = key, score = 0.07660000000000002
np.mean(results), mse, len(results) =  0.0766307789740342 7708.8354 1579
slice = train, score = 0.0766307789740342
np.mean(results), mse, len(results) =  0.07423611111111111 7720.7812 720
slice = test, score = 0.07423611111111111

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.1866 1735645.8 100
slice = key, score = 0.1866
np.mean(results), mse, len(results) =  0.1924889170360988 1042398.3 1579
slice = train, score = 0.1924889170360988
np.mean(results), mse, len(results) =  0.18458333333333335 1131800.6 720
slice = test, score = 0.18458333333333335

=== Iteration 2000 ===
np.mean(results), mse, 

In [29]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )

    N = 100000
    m = CBKnnV0(ctx, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': True,
        'train_popbias': True, 'train_bias': True,
        'verbose': False, 'loss': 'softmax_signed',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 1000,
        'trainable_items': True  # <<< DIFF HERE
    })
    m.fit()
    m.get_score("train"), m.get_score("test")
    
    del ctx
    gc.collect()

    del m
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
self.game

np.mean(results), mse, len(results) =  0.6025486725663717 89480.1 2260
slice = train, score = 0.6025486725663717
np.mean(results), mse, len(results) =  0.4949160908193485 85578.78 1013
slice = test, score = 0.4949160908193485

=== Iteration 21000 ===
np.mean(results), mse, len(results) =  0.5681 108958.46 100
slice = key, score = 0.5681
np.mean(results), mse, len(results) =  0.6048185840707964 94109.664 2260
slice = train, score = 0.6048185840707964
np.mean(results), mse, len(results) =  0.494106614017769 91345.95 1013
slice = test, score = 0.494106614017769

=== Iteration 22000 ===
np.mean(results), mse, len(results) =  0.5781 94403.35 100
slice = key, score = 0.5781
np.mean(results), mse, len(results) =  0.6083893805309735 88488.03 2260
slice = train, score = 0.6083893805309735
np.mean(results), mse, len(results) =  0.49614017769002966 83254.46 1013
slice = test, score = 0.49614017769002966

=== Iteration 23000 ===
np.mean(results), mse, len(results) =  0.5698000000000001 141858.47 1

np.mean(results), mse, len(results) =  0.5031490621915103 202571.78 1013
slice = test, score = 0.5031490621915103

=== Iteration 44000 ===
np.mean(results), mse, len(results) =  0.5615000000000001 277248.0 100
slice = key, score = 0.5615000000000001
np.mean(results), mse, len(results) =  0.6364380530973451 254789.86 2260
slice = train, score = 0.6364380530973451
np.mean(results), mse, len(results) =  0.5016979269496545 243052.9 1013
slice = test, score = 0.5016979269496545

=== Iteration 45000 ===
np.mean(results), mse, len(results) =  0.5746999999999999 290914.53 100
slice = key, score = 0.5746999999999999
np.mean(results), mse, len(results) =  0.6410840707964601 257799.98 2260
slice = train, score = 0.6410840707964601
np.mean(results), mse, len(results) =  0.5059131293188549 240260.27 1013
slice = test, score = 0.5059131293188549

=== Iteration 46000 ===
np.mean(results), mse, len(results) =  0.5726 265815.06 100
slice = key, score = 0.5726
np.mean(results), mse, len(results) =  0.64


=== Iteration 67000 ===
np.mean(results), mse, len(results) =  0.5632 511512.4 100
slice = key, score = 0.5632
np.mean(results), mse, len(results) =  0.6552920353982301 448634.25 2260
slice = train, score = 0.6552920353982301
np.mean(results), mse, len(results) =  0.5042053307008885 431025.44 1013
slice = test, score = 0.5042053307008885

=== Iteration 68000 ===
np.mean(results), mse, len(results) =  0.5649000000000001 562454.9 100
slice = key, score = 0.5649000000000001
np.mean(results), mse, len(results) =  0.6563982300884956 496004.03 2260
slice = train, score = 0.6563982300884956
np.mean(results), mse, len(results) =  0.5049062191510365 467462.97 1013
slice = test, score = 0.5049062191510365

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.5711999999999999 466982.62 100
slice = key, score = 0.5711999999999999
np.mean(results), mse, len(results) =  0.6588407079646018 424659.75 2260
slice = train, score = 0.6588407079646018
np.mean(results), mse, len(results) =  0.5

np.mean(results), mse, len(results) =  0.6660530973451328 588468.5 2260
slice = train, score = 0.6660530973451328
np.mean(results), mse, len(results) =  0.5068015794669299 555885.1 1013
slice = test, score = 0.5068015794669299

=== Iteration 91000 ===
np.mean(results), mse, len(results) =  0.5599 672691.25 100
slice = key, score = 0.5599
np.mean(results), mse, len(results) =  0.666787610619469 619948.94 2260
slice = train, score = 0.666787610619469
np.mean(results), mse, len(results) =  0.5071569595261599 574568.8 1013
slice = test, score = 0.5071569595261599

=== Iteration 92000 ===
np.mean(results), mse, len(results) =  0.5618 700440.56 100
slice = key, score = 0.5618
np.mean(results), mse, len(results) =  0.6687477876106194 636577.1 2260
slice = train, score = 0.6687477876106194
np.mean(results), mse, len(results) =  0.5067423494570582 627981.1 1013
slice = test, score = 0.5067423494570582

=== Iteration 93000 ===
np.mean(results), mse, len(results) =  0.5501 816665.75 100
slice = k

...

np.mean(results), mse, len(results) =  0.24752500926268986 6922193.0 2699
slice = train, score = 0.24752500926268986
np.mean(results), mse, len(results) =  0.19590833333333332 7159416.0 1200
slice = test, score = 0.19590833333333332

=== Iteration 98000 ===
np.mean(results), mse, len(results) =  0.2425 8021005.0 100
slice = key, score = 0.2425
np.mean(results), mse, len(results) =  0.25393849573916266 6233869.0 2699
slice = train, score = 0.25393849573916266
np.mean(results), mse, len(results) =  0.198325 6714696.5 1200
slice = test, score = 0.198325

=== Iteration 99000 ===
np.mean(results), mse, len(results) =  0.2416 7808473.5 100
slice = key, score = 0.2416
np.mean(results), mse, len(results) =  0.2577361985920712 5996724.5 2699
slice = train, score = 0.2577361985920712
np.mean(results), mse, len(results) =  0.19988333333333336 6370434.5 1200
slice = test, score = 0.19988333333333336
last loss =  tf.Tensor(-0.0032985606, shape=(), dtype=float32)
np.mean(results), mse, len(results) 


=== Iteration 13000 ===
np.mean(results), mse, len(results) =  0.19310000000000002 20773948.0 100
slice = key, score = 0.19310000000000002
np.mean(results), mse, len(results) =  0.2129512349588347 14851754.0 1579
slice = train, score = 0.2129512349588347
np.mean(results), mse, len(results) =  0.19795833333333335 15645942.0 720
slice = test, score = 0.19795833333333335

=== Iteration 14000 ===
np.mean(results), mse, len(results) =  0.19079999999999997 22178486.0 100
slice = key, score = 0.19079999999999997
np.mean(results), mse, len(results) =  0.20747308423052566 16122868.0 1579
slice = train, score = 0.20747308423052566
np.mean(results), mse, len(results) =  0.19320833333333332 16888614.0 720
slice = test, score = 0.19320833333333332

=== Iteration 15000 ===
np.mean(results), mse, len(results) =  0.1924 23376248.0 100
slice = key, score = 0.1924
np.mean(results), mse, len(results) =  0.21115262824572512 16598342.0 1579
slice = train, score = 0.21115262824572512
np.mean(results), mse,

np.mean(results), mse, len(results) =  0.19866666666666669 42364172.0 720
slice = test, score = 0.19866666666666669

=== Iteration 36000 ===
np.mean(results), mse, len(results) =  0.1931 56190896.0 100
slice = key, score = 0.1931
np.mean(results), mse, len(results) =  0.21743508549715007 42117980.0 1579
slice = train, score = 0.21743508549715007
np.mean(results), mse, len(results) =  0.1977638888888889 42600972.0 720
slice = test, score = 0.1977638888888889

=== Iteration 37000 ===
np.mean(results), mse, len(results) =  0.19389999999999996 60134748.0 100
slice = key, score = 0.19389999999999996
np.mean(results), mse, len(results) =  0.21943635212159596 43770920.0 1579
slice = train, score = 0.21943635212159596
np.mean(results), mse, len(results) =  0.2030138888888889 46194204.0 720
slice = test, score = 0.2030138888888889

=== Iteration 38000 ===
np.mean(results), mse, len(results) =  0.18209999999999998 65217540.0 100
slice = key, score = 0.18209999999999998
np.mean(results), mse, len

np.mean(results), mse, len(results) =  0.2121659278024066 63981790.0 1579
slice = train, score = 0.2121659278024066
np.mean(results), mse, len(results) =  0.19316666666666668 65975730.0 720
slice = test, score = 0.19316666666666668

=== Iteration 59000 ===
np.mean(results), mse, len(results) =  0.1852 96403570.0 100
slice = key, score = 0.1852
np.mean(results), mse, len(results) =  0.21691576947435084 70053860.0 1579
slice = train, score = 0.21691576947435084
np.mean(results), mse, len(results) =  0.19712500000000002 72471410.0 720
slice = test, score = 0.19712500000000002

=== Iteration 60000 ===
np.mean(results), mse, len(results) =  0.18650000000000003 98374150.0 100
slice = key, score = 0.18650000000000003
np.mean(results), mse, len(results) =  0.21823939202026602 70461050.0 1579
slice = train, score = 0.21823939202026602
np.mean(results), mse, len(results) =  0.196625 72877860.0 720
slice = test, score = 0.196625

=== Iteration 61000 ===
np.mean(results), mse, len(results) =  0.17

np.mean(results), mse, len(results) =  0.21390753641545282 106918130.0 1579
slice = train, score = 0.21390753641545282
np.mean(results), mse, len(results) =  0.19048611111111108 109621624.0 720
slice = test, score = 0.19048611111111108

=== Iteration 82000 ===
np.mean(results), mse, len(results) =  0.18290000000000003 129412200.0 100
slice = key, score = 0.18290000000000003
np.mean(results), mse, len(results) =  0.22102596580113995 94605340.0 1579
slice = train, score = 0.22102596580113995
np.mean(results), mse, len(results) =  0.1965694444444444 96840650.0 720
slice = test, score = 0.1965694444444444

=== Iteration 83000 ===
np.mean(results), mse, len(results) =  0.1804 147000880.0 100
slice = key, score = 0.1804
np.mean(results), mse, len(results) =  0.21835338822039263 111466810.0 1579
slice = train, score = 0.21835338822039263
np.mean(results), mse, len(results) =  0.19697222222222224 114072696.0 720
slice = test, score = 0.19697222222222224

=== Iteration 84000 ===
np.mean(results

### games

In [12]:
L = 7000
N = 1000
DA = 50

In [33]:
!ls -lah

total 3.9G
drwxr-xr-x 10 shevkunov shevkunov 4.0K Sep  6 11:48 .
drwxr-xr-x 56 shevkunov shevkunov 4.0K May  6 01:49 ..
drwxrwxr-x  2 shevkunov shevkunov 4.0K Aug 20 15:53 doctor_who
-rwxr-xr-x  1 shevkunov shevkunov 108M Apr 14 15:02 games-all.json
drwxr-xr-x  2 shevkunov shevkunov 4.0K Aug 24 19:45 .ipynb_checkpoints
-rw-rw-r--  1 shevkunov shevkunov 1.3G May 19 12:26 log.local.bin
-rwxr-xr-x  1 shevkunov shevkunov 691M May 19 11:32 log.local.bin-old
-rw-rw-r--  1 shevkunov shevkunov 1.8G Jul 22 00:21 log.local.logtime2.bin
-rw-r--r--  1 shevkunov shevkunov 5.5K Jul  9 17:54 matrix_approx_zeshel.py
drwxrwxr-x  2 shevkunov shevkunov 4.0K Sep  2 17:08 military
-rwxr-xr-x  1 shevkunov shevkunov 345K Feb  8  2023 proof-of-concept.ipynb
-rw-rw-r--  1 shevkunov shevkunov 438K Jul 12 12:23 proof-of-concept-open-data-round1.ipynb
-rw-rw-r--  1 shevkunov shevkunov 898K Jul 30 16:51 proof-of-concept-open-data-round2-anncurcomparison.ipynb
-rw-rw-r--  1 shevkunov shevkunov  60K Aug 24 13:30 pro

In [13]:
ctx = load(L, raw_path = "stand/log.local.txt", seed=17, det_attempts=DA) # .set_top_games_as_key()
ev([
    AnnCUR(ctx)
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.628053728316076e-117
2.628053728316076e-117
101 4644 2034



model =  AnnCur(139929851077648)
np.mean(results), mse, len(results) =  0.6764900947459087 1.9841502796717772 4644
np.mean(results), mse, len(results) =  0.6652409046214356 2.8454261374203558 2034
0.6764900947459087 0.6652409046214356


In [16]:
ctx = load(L, raw_path = "stand/log.local.txt", seed=17, det_attempts=DA).set_top_games_as_key()

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.628053728316076e-117
2.628053728316076e-117
101 4644 2034


In [27]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(





model =  <__main__.Popular object at 0x7fd33860bb20>
np.mean(results), mse, len(results) =  0.5083527131782946 2.1090857474029794 4644
np.mean(results), mse, len(results) =  0.5076548672566372 2.214983388749716 2034
0.5083527131782946 0.5076548672566372



model =  AnnCur(140545160627632)
np.mean(results), mse, len(results) =  0.7693260120585702 0.5653017362404418 4644
np.mean(results), mse, len(results) =  0.762251720747296 0.5927785494345469 2034
0.7693260120585702 0.762251720747296



model =  AnnCur(key_games=np.arange(100) - 140531535266816)
np.mean(results), mse, len(results) =  0.6579565030146425 1.9767935375740953 4644
np.mean(results), mse, len(results) =  0.6489528023598821 6.906013812566882 2034
0.6579565030146425 0.6489528023598821



model =  AnnCur(random - 140531535278288)
np.mean(results), mse, len(results) =  0.6810099052540913 1.9702658686057843 4644
np.mean(results), mse, len(results) =  0.6697246804326451 2.180510915221889 2034
0.6810099052540913 0.66972468043264

In [34]:
!ls stand

log.local.logtime2.txt.old  log.local.txt  log.local.txt.tar.gz


In [13]:
ctx = load(L, raw_path = "stand/log.local.logtime2.txt",
           path="log.local.logtime2.true.bin", seed=17, det_attempts=DA)

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.609521994482665e-120
2.609521994482665e-120
101 4769 2088


In [14]:
ev([
    Popular(ctx),
    AnnCUR(ctx)
])




model =  <__main__.Popular object at 0x7ff62130f6d0>
np.mean(results), mse, len(results) =  0.2893499685468652 5.200994032516979 4769
np.mean(results), mse, len(results) =  0.2885536398467433 5.111057961798107 2088
0.2893499685468652 0.2885536398467433



model =  AnnCur(140695785624080)
np.mean(results), mse, len(results) =  0.5976535961417488 0.17482987219177043 4769
np.mean(results), mse, len(results) =  0.5860488505747127 0.19354859381230788 2088
0.5976535961417488 0.5860488505747127


In [15]:
ctx = load(L, raw_path = "stand/log.local.logtime2.txt",
           path="log.local.logtime2.true.bin", seed=17, det_attempts=DA).set_top_games_as_key()

/tmp/ipykernel_690140/664571995.py:33: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for i, line in tqdm.tqdm_notebook(enumerate(f)):


0it [00:00, ?it/s]

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.609521994482665e-120
2.609521994482665e-120
101 4769 2088


In [16]:
ev([
    Popular(ctx),
    AnnCUR(ctx)
])




model =  <__main__.Popular object at 0x7f3894f410c0>
np.mean(results), mse, len(results) =  0.2893499685468652 5.200994032516979 4769
np.mean(results), mse, len(results) =  0.2885536398467433 5.111057961798107 2088
0.2893499685468652 0.2885536398467433



model =  AnnCur(139872009046960)
np.mean(results), mse, len(results) =  0.6778381211994128 0.24971886538105065 4769
np.mean(results), mse, len(results) =  0.669477969348659 0.27141239614050905 2088
0.6778381211994128 0.669477969348659


In [17]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(





model =  <__main__.Popular object at 0x7f3677c5d450>
np.mean(results), mse, len(results) =  0.2893499685468652 5.200994032516979 4769
np.mean(results), mse, len(results) =  0.2885536398467433 5.111057961798107 2088
0.2893499685468652 0.2885536398467433



model =  AnnCur(139880993919168)
np.mean(results), mse, len(results) =  0.6778381211994128 0.24971886538105065 4769
np.mean(results), mse, len(results) =  0.669477969348659 0.27141239614050905 2088
0.6778381211994128 0.669477969348659



model =  AnnCur(key_games=np.arange(100) - 139886535450944)
np.mean(results), mse, len(results) =  0.568666387083246 0.2101395930999089 4769
np.mean(results), mse, len(results) =  0.5609003831417625 0.2306177104605779 2088
0.568666387083246 0.5609003831417625



model =  AnnCur(random - 139886535448400)
np.mean(results), mse, len(results) =  0.5932228978821555 0.1733168362865605 4769
np.mean(results), mse, len(results) =  0.584176245210728 0.1904597187905242 2088
0.5932228978821555 0.58417624521072

In [ ]:
ctx = load(L, raw_path = "stand/log.local.logtime2.txt",
           path="log.local.logtime2.true.bin", seed=17, det_attempts=DA)

ctx.set_kmeans_games_as_key()

N = 20000

m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': False,
    'verbose': False, 'loss': 'softmax',
    'loss_batch': 128, 'loss_q': 0.99,
    'n': N,
    # 'ubatch': 512
    'score_verbose': 1000,
    'trainable_items': True  # <<< DIFF HERE
})
m.fit()
m.get_score("train"), m.get_score("test")


del ctx
gc.collect()

del m
gc.collect()

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.609521994482665e-120
2.609521994482665e-120
101 4769 2088


/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


self.embed_users['train'].shape =  (4769, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (4769, 101)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.13990099009900994 4282.683400224812 101
slice = key, score = 0.13990099009900994


In [ ]:
ctx = load(L, raw_path = "stand/log.local.logtime2.txt",
           path="log.local.logtime2.true.bin", seed=17, det_attempts=DA)

ctx.set_kmeans_games_as_key()

N = 20000

m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': False,
    'verbose': False, 'loss': 'softmax',
    'loss_batch': 128, 'loss_q': 0.99,
    'n': N,
    # 'ubatch': 512
    'score_verbose': 1000,
    'trainable_items': True  # <<< DIFF HERE
})
m.fit()
m.get_score("train"), m.get_score("test")


del ctx
gc.collect()

del m
gc.collect()